In [26]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings


import matplotlib.pyplot as plt
import seaborn as sns
HAS_VIZ = True
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [27]:
def load_data():
    """Load the features CSV file."""
    data_path =  Path("../data/processed/Barcelona_2014_2015_features.csv")
    
    if not data_path.exists():
        raise FileNotFoundError(f"Data file not found: {data_path}")
    
    df = pd.read_csv(data_path)
    print("=" * 80)
    print("CELL 1: DATA LOADING AND BASIC INFORMATION")
    print("=" * 80)
    print(f"\n✓ Data loaded successfully from: {data_path}")
    print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumn Names ({len(df.columns)}):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    return df

df=load_data()

CELL 1: DATA LOADING AND BASIC INFORMATION

✓ Data loaded successfully from: ../data/processed/Barcelona_2014_2015_features.csv

Dataset Shape: 195 rows × 74 columns

Column Names (74):
   1. player_id
   2. player_name
   3. midfielder_type
   4. match_id
   5. team_id
   6. team_name
   7. season
   8. computed_at
   9. possessions_involved
  10. possession_time_seconds
  11. tempo_index
  12. turnovers
  13. passes_attempted
  14. pass_completion_rate
  15. progressive_passes
  16. final_third_entries_by_pass
  17. key_passes
  18. under_pressure_pass_share
  19. carries_attempted
  20. progressive_carries
  21. carry_distance_total
  22. successful_dribbles
  23. carries_leading_to_shot
  24. carries_leading_to_key_pass
  25. final_third_carries
  26. penalty_area_carries
  27. pressured_carry_success_rate
  28. pressures_applied
  29. ball_recoveries
  30. interceptions
  31. tackles_won
  32. press_to_interception_chain
  33. counterpress_actions
  34. pressure_to_self_recovery
 

In [28]:
def analyze_missing_values(df):
    """Analyze missing values in the dataset."""
    print("\n" + "=" * 80)
    print("CELL 2: MISSING VALUES ANALYSIS")
    print("=" * 80)
    
    # Identify metadata vs feature columns
    metadata_cols = ['player_id', 'player_name', 'midfielder_type', 'match_id', 
                     'team_id', 'team_name', 'season', 'computed_at']
    feature_cols = [c for c in df.columns if c not in metadata_cols]
    
    print(f"\nMetadata columns: {len(metadata_cols)}")
    print(f"Feature columns: {len(feature_cols)}")
    
    # Missing values analysis
    missing = df[feature_cols].isna().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_pct
    }).sort_values('Missing Count', ascending=False)
    
    print("\nMissing Values Summary:")
    print("-" * 80)
    missing_nonzero = missing_df[missing_df['Missing Count'] > 0]
    if len(missing_nonzero) > 0:
        print(missing_nonzero.to_string())
    else:
        print("  ✓ No missing values found!")
    
    # Visualize missing values
    if HAS_VIZ and len(missing_nonzero) > 0:
        plt.figure(figsize=(12, 8))
        missing_nonzero['Missing %'].plot(kind='barh')
        plt.xlabel('Missing Percentage (%)')
        plt.title('Missing Values by Feature')
        plt.tight_layout()
        plt.savefig(Path("../data/processed/eda_missing_values.png"))
        print(f"\n✓ Missing values plot saved to: data/processed/eda_missing_values.png")
        plt.close()
    elif not HAS_VIZ and len(missing_nonzero) > 0:
        print("\n⚠ Visualization skipped (matplotlib not available)")
    
    return metadata_cols, feature_cols

metadata_cols, feature_cols = analyze_missing_values(df)


CELL 2: MISSING VALUES ANALYSIS

Metadata columns: 8
Feature columns: 66

Missing Values Summary:
--------------------------------------------------------------------------------
                             Missing Count  Missing %
cross_accuracy                         145  74.358974
corner_delivery_accuracy               135  69.230769
sliding_tackle_success_rate            122  62.564103
expected_assists                        79  40.512821
under_pressure_pass_share                3   1.538462
weak_foot_pass_share                     1   0.512821



✓ Missing values plot saved to: data/processed/eda_missing_values.png


In [29]:
def descriptive_statistics(df, feature_cols):
    """Generate descriptive statistics for all features."""
    print("\n" + "=" * 80)
    print("CELL 3: DESCRIPTIVE STATISTICS")
    print("=" * 80)
    
    # Basic statistics
    numeric_features = df[feature_cols].select_dtypes(include=[np.number])
    
    print(f"\nNumeric Features: {len(numeric_features.columns)}")
    print("\nSummary Statistics:")
    print("-" * 80)
    stats = numeric_features.describe().T
    stats['range'] = stats['max'] - stats['min']
    stats = stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'range']]
    print(stats.round(2).to_string())
    
    # Save to CSV
    stats_path = Path("../data/processed/eda_descriptive_stats.csv")
    stats.to_csv(stats_path)
    print(f"\n✓ Full statistics saved to: {stats_path}")
    
    return numeric_features

numeric_features = descriptive_statistics(df, feature_cols)


CELL 3: DESCRIPTIVE STATISTICS

Numeric Features: 66

Summary Statistics:
--------------------------------------------------------------------------------
                                count    mean     std     min     25%     50%     75%     max   range
possessions_involved            195.0   58.32   12.34   20.00   50.50   60.00   67.50   89.00   69.00
possession_time_seconds         195.0  352.63  148.25   53.94  261.60  332.77  416.34  879.55  825.61
tempo_index                     195.0   21.55    3.57   13.75   19.32   21.35   22.97   43.38   29.63
turnovers                       195.0    2.71    2.60    0.00    1.00    2.00    4.00   12.00   12.00
passes_attempted                195.0   63.17   23.63   14.00   47.00   63.00   78.00  132.00  118.00
pass_completion_rate            195.0    0.86    0.08    0.58    0.82    0.88    0.92    1.00    0.42
progressive_passes              195.0   15.16    8.84    0.00    8.00   15.00   21.50   41.00   41.00
final_third_entries_by_pass 

In [30]:

def analyze_midfielder_types(df):
    """Analyze distribution and characteristics by midfielder type."""
    print("\n" + "=" * 80)
    print("CELL 4: MIDFIELDER TYPE ANALYSIS")
    print("=" * 80)
    
    type_names = {
        0: 'Defensive Midfield',
        1: 'Center Midfield',
        2: 'Attacking Midfield',
        3: 'Wing Midfield',
        4: 'Right Wing',
        5: 'Left Wing',
        6: 'Wing Back',
        7: 'Midfield (Generic)'
    }
    
    df['midfielder_type_name'] = df['midfielder_type'].map(type_names)
    
    print("\nMidfielder Type Distribution:")
    print("-" * 80)
    type_counts = df['midfielder_type_name'].value_counts().sort_index()
    for type_code, count in type_counts.items():
        pct = (count / len(df)) * 100
        print(f"  {type_code:30s}: {count:3d} ({pct:5.1f}%)")
    
    # Key features by type
    key_features = ['passes_attempted', 'pass_completion_rate', 'carries_attempted',
                    'pressures_applied', 'ball_recoveries', 'average_position_x', 
                    'average_position_y']
    
    print("\n\nAverage Key Features by Midfielder Type:")
    print("-" * 80)
    type_means = df.groupby('midfielder_type_name')[key_features].mean().round(2)
    print(type_means.to_string())
    
    # Visualization
    if not HAS_VIZ:
        print("\n⚠ Visualization skipped (matplotlib not available)")
        return type_names
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Type distribution
    type_counts.plot(kind='bar', ax=axes[0, 0], color='steelblue')
    axes[0, 0].set_title('Midfielder Type Distribution')
    axes[0, 0].set_xlabel('Midfielder Type')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Passes by type
    df.boxplot(column='passes_attempted', by='midfielder_type_name', ax=axes[0, 1])
    axes[0, 1].set_title('Passes Attempted by Type')
    axes[0, 1].set_xlabel('')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Position heatmap
    position_data = df.groupby('midfielder_type_name')[['average_position_x', 'average_position_y']].mean()
    axes[1, 0].scatter(position_data['average_position_x'], position_data['average_position_y'], 
                       s=200, alpha=0.6)
    for idx, row in position_data.iterrows():
        axes[1, 0].annotate(idx, (row['average_position_x'], row['average_position_y']))
    axes[1, 0].set_xlabel('Average Position X')
    axes[1, 0].set_ylabel('Average Position Y')
    axes[1, 0].set_title('Average Position by Midfielder Type')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Pressures by type
    df.boxplot(column='pressures_applied', by='midfielder_type_name', ax=axes[1, 1])
    axes[1, 1].set_title('Pressures Applied by Type')
    axes[1, 1].set_xlabel('')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(Path("../data/processed/eda_midfielder_types.png"))
    print(f"\n✓ Midfielder type analysis plot saved to: data/processed/eda_midfielder_types.png")
    plt.close()
    
    return type_names


type_names = analyze_midfielder_types(df)


CELL 4: MIDFIELDER TYPE ANALYSIS

Midfielder Type Distribution:
--------------------------------------------------------------------------------
  Center Midfield               :  78 ( 40.0%)
  Defensive Midfield            :  39 ( 20.0%)
  Left Wing                     :  39 ( 20.0%)
  Right Wing                    :  39 ( 20.0%)


Average Key Features by Midfielder Type:
--------------------------------------------------------------------------------
                      passes_attempted  pass_completion_rate  carries_attempted  pressures_applied  ball_recoveries  average_position_x  average_position_y
midfielder_type_name                                                                                                                                       
Center Midfield                  70.53                  0.88               8.71              12.00             4.73               68.95               39.63
Defensive Midfield               72.15                  0.92              

In [32]:
def analyze_feature_distributions(df, feature_cols):
    """Analyze distributions of key features."""
    print("\n" + "=" * 80)
    print("CELL 5: FEATURE DISTRIBUTIONS")
    print("=" * 80)
    
    key_features = ['passes_attempted', 'pass_completion_rate', 'carries_attempted',
                    'pressures_applied', 'ball_recoveries', 'possessions_involved',
                    'possession_time_seconds', 'progressive_passes', 'progressive_carries']
    
    available_features = [f for f in key_features if f in feature_cols]
    
    # Create distribution plots
    if not HAS_VIZ:
        print("\n⚠ Visualization skipped (matplotlib not available)")
        return
    
    n_features = len(available_features)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
    axes = axes.flatten() if n_features > 1 else [axes]
    
    for idx, feature in enumerate(available_features):
        ax = axes[idx]
        data = df[feature].dropna()
        
        if len(data) > 0:
            ax.hist(data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
            ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {data.mean():.2f}')
            ax.axvline(data.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {data.median():.2f}')
            ax.set_xlabel(feature.replace('_', ' ').title())
            ax.set_ylabel('Frequency')
            ax.set_title(f'Distribution of {feature.replace("_", " ").title()}')
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(n_features, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(Path("../data/processed/eda_feature_distributions.png"))
    print(f"\n✓ Feature distributions plot saved to: data/processed/eda_feature_distributions.png")
    plt.close()
    
    # Print skewness and kurtosis
    print("\nFeature Skewness and Kurtosis:")
    print("-" * 80)
    skew_kurt = pd.DataFrame({
        'Skewness': [df[f].skew() for f in available_features],
        'Kurtosis': [df[f].kurtosis() for f in available_features]
    }, index=available_features)
    print(skew_kurt.round(2).to_string())

analyze_feature_distributions(df, feature_cols)


CELL 5: FEATURE DISTRIBUTIONS

✓ Feature distributions plot saved to: data/processed/eda_feature_distributions.png

Feature Skewness and Kurtosis:
--------------------------------------------------------------------------------
                         Skewness  Kurtosis
passes_attempted             0.29      0.03
pass_completion_rate        -0.81      0.40
carries_attempted            0.87      0.42
pressures_applied            0.73      0.94
ball_recoveries              0.36     -0.33
possessions_involved        -0.41      0.20
possession_time_seconds      0.86      1.20
progressive_passes           0.35     -0.46
progressive_carries          0.54     -0.50


In [33]:

def analyze_correlations(df, feature_cols):
    """Analyze correlations between features."""
    print("\n" + "=" * 80)
    print("CELL 6: CORRELATION ANALYSIS")
    print("=" * 80)
    
    # Select numeric features
    numeric_features = df[feature_cols].select_dtypes(include=[np.number])
    
    # Calculate correlation matrix
    corr_matrix = numeric_features.corr()
    
    # Find highly correlated pairs
    print("\nHighly Correlated Feature Pairs (|r| > 0.7):")
    print("-" * 80)
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.7 and not np.isnan(corr_val):
                high_corr_pairs.append((
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_val
                ))
    
    if high_corr_pairs:
        high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])
        high_corr_df = high_corr_df.sort_values('Correlation', key=abs, ascending=False)
        print(high_corr_df.to_string(index=False))
    else:
        print("  No highly correlated pairs found (|r| > 0.7)")
    
    # Visualize correlation matrix for key features
    key_features = ['passes_attempted', 'pass_completion_rate', 'carries_attempted',
                    'pressures_applied', 'ball_recoveries', 'possessions_involved',
                    'possession_time_seconds', 'progressive_passes', 'progressive_carries',
                    'average_position_x', 'average_position_y']
    
    available_key = [f for f in key_features if f in numeric_features.columns]
    
    if len(available_key) > 1:
        key_corr = numeric_features[available_key].corr()
        
        if HAS_VIZ:
            plt.figure(figsize=(14, 12))
            mask = np.triu(np.ones_like(key_corr, dtype=bool))
            sns.heatmap(key_corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
                        center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
            plt.title('Correlation Matrix: Key Features')
            plt.tight_layout()
            plt.savefig(Path("../data/processed/eda_correlation_matrix.png"))
            print(f"\n✓ Correlation matrix plot saved to: data/processed/eda_correlation_matrix.png")
            plt.close()
        else:
            print("\n⚠ Visualization skipped (matplotlib not available)")
    
    # Save full correlation matrix
    corr_path = Path("../data/processed/eda_correlation_matrix.csv")
    corr_matrix.to_csv(corr_path)
    print(f"✓ Full correlation matrix saved to: {corr_path}")

analyze_correlations(df, feature_cols)


CELL 6: CORRELATION ANALYSIS

Highly Correlated Feature Pairs (|r| > 0.7):
--------------------------------------------------------------------------------
                     Feature 1                      Feature 2  Correlation
pressured_touch_retention_rate       pressured_retention_rate     1.000000
             carries_attempted           carry_distance_total     0.974639
                  zone_entries          central_lane_receipts     0.965050
         shot_creating_actions         secondary_shot_assists     0.940135
  pressured_carry_success_rate       pressured_retention_rate     0.918604
  pressured_carry_success_rate pressured_touch_retention_rate     0.918604
       possession_time_seconds            ball_receipts_total     0.889691
       possession_time_seconds               passes_attempted     0.886887
          carry_distance_total            final_third_carries     0.872041
             carries_attempted            final_third_carries     0.869499
           progres

In [7]:

def analyze_feature_relationships(df):
    """Analyze relationships between key features."""
    print("\n" + "=" * 80)
    print("CELL 7: FEATURE RELATIONSHIPS")
    print("=" * 80)
    
    # Scatter plots for key relationships
    if not HAS_VIZ:
        print("\n⚠ Visualization skipped (matplotlib not available)")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Passes vs Carries
    axes[0, 0].scatter(df['passes_attempted'], df['carries_attempted'], 
                       alpha=0.6, s=50, c=df['midfielder_type'], cmap='viridis')
    axes[0, 0].set_xlabel('Passes Attempted')
    axes[0, 0].set_ylabel('Carries Attempted')
    axes[0, 0].set_title('Passes vs Carries (colored by midfielder type)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Pass completion rate vs Pressures
    axes[0, 1].scatter(df['pass_completion_rate'], df['pressures_applied'],
                       alpha=0.6, s=50, c=df['midfielder_type'], cmap='viridis')
    axes[0, 1].set_xlabel('Pass Completion Rate')
    axes[0, 1].set_ylabel('Pressures Applied')
    axes[0, 1].set_title('Pass Completion vs Pressures')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Possessions vs Time
    axes[1, 0].scatter(df['possessions_involved'], df['possession_time_seconds'],
                      alpha=0.6, s=50, c=df['midfielder_type'], cmap='viridis')
    axes[1, 0].set_xlabel('Possessions Involved')
    axes[1, 0].set_ylabel('Possession Time (seconds)')
    axes[1, 0].set_title('Possessions vs Time')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Position scatter
    axes[1, 1].scatter(df['average_position_x'], df['average_position_y'],
                      alpha=0.6, s=50, c=df['midfielder_type'], cmap='viridis')
    axes[1, 1].set_xlabel('Average Position X')
    axes[1, 1].set_ylabel('Average Position Y')
    axes[1, 1].set_title('Average Position on Pitch (colored by type)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(Path("../data/processed/eda_feature_relationships.png"))
    print(f"\n✓ Feature relationships plot saved to: data/processed/eda_feature_relationships.png")
    plt.close()

analyze_feature_relationships(df)


CELL 7: FEATURE RELATIONSHIPS

✓ Feature relationships plot saved to: data/processed/eda_feature_relationships.png


In [8]:

def analyze_players(df):
    """Analyze individual player performance."""
    print("\n" + "=" * 80)
    print("CELL 8: PLAYER-LEVEL ANALYSIS")
    print("=" * 80)
    
    # Aggregate by player
    player_stats = df.groupby('player_name').agg({
        'match_id': 'count',
        'passes_attempted': 'mean',
        'pass_completion_rate': 'mean',
        'carries_attempted': 'mean',
        'pressures_applied': 'mean',
        'ball_recoveries': 'mean',
        'possessions_involved': 'mean',
        'possession_time_seconds': 'mean',
        'progressive_passes': 'mean',
        'progressive_carries': 'mean'
    }).round(2)
    
    player_stats.columns = ['Matches', 'Avg_Passes', 'Avg_Pass_Comp_Rate', 'Avg_Carries',
                           'Avg_Pressures', 'Avg_Recoveries', 'Avg_Possessions',
                           'Avg_Possession_Time', 'Avg_Prog_Passes', 'Avg_Prog_Carries']
    player_stats = player_stats.sort_values('Matches', ascending=False)
    
    print("\nPlayer Performance Summary (sorted by number of matches):")
    print("-" * 80)
    print(player_stats.to_string())
    
    # Top performers
    print("\n\nTop 5 Players by Key Metrics:")
    print("-" * 80)
    
    metrics = {
        'Most Passes': 'Avg_Passes',
        'Best Pass Accuracy': 'Avg_Pass_Comp_Rate',
        'Most Pressures': 'Avg_Pressures',
        'Most Recoveries': 'Avg_Recoveries',
        'Most Progressive Passes': 'Avg_Prog_Passes'
    }
    
    for metric_name, metric_col in metrics.items():
        top_5 = player_stats.nlargest(5, metric_col)[[metric_col]]
        print(f"\n{metric_name}:")
        print(top_5.to_string())
    
    # Save player stats
    player_stats_path = Path("../data/processed/eda_player_stats.csv")
    player_stats.to_csv(player_stats_path)
    print(f"\n✓ Player statistics saved to: {player_stats_path}")
    
    # Visualization
    if not HAS_VIZ:
        print("\n⚠ Visualization skipped (matplotlib not available)")
        return
    
    top_players = player_stats.head(10)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Passes and carries
    x = np.arange(len(top_players))
    width = 0.35
    axes[0, 0].bar(x - width/2, top_players['Avg_Passes'], width, label='Passes', alpha=0.8)
    axes[0, 0].bar(x + width/2, top_players['Avg_Carries'], width, label='Carries', alpha=0.8)
    axes[0, 0].set_xlabel('Player')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Top 10 Players: Passes vs Carries')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(top_players.index, rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Pass completion rate
    axes[0, 1].barh(range(len(top_players)), top_players['Avg_Pass_Comp_Rate'], alpha=0.8)
    axes[0, 1].set_yticks(range(len(top_players)))
    axes[0, 1].set_yticklabels(top_players.index)
    axes[0, 1].set_xlabel('Pass Completion Rate')
    axes[0, 1].set_title('Top 10 Players: Pass Completion Rate')
    axes[0, 1].grid(True, alpha=0.3, axis='x')
    
    # Pressures
    axes[1, 0].barh(range(len(top_players)), top_players['Avg_Pressures'], alpha=0.8, color='orange')
    axes[1, 0].set_yticks(range(len(top_players)))
    axes[1, 0].set_yticklabels(top_players.index)
    axes[1, 0].set_xlabel('Pressures Applied')
    axes[1, 0].set_title('Top 10 Players: Pressures Applied')
    axes[1, 0].grid(True, alpha=0.3, axis='x')
    
    # Progressive actions
    axes[1, 1].bar(x - width/2, top_players['Avg_Prog_Passes'], width, label='Prog Passes', alpha=0.8)
    axes[1, 1].bar(x + width/2, top_players['Avg_Prog_Carries'], width, label='Prog Carries', alpha=0.8)
    axes[1, 1].set_xlabel('Player')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Top 10 Players: Progressive Actions')
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(top_players.index, rotation=45, ha='right')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(Path("../data/processed/eda_player_analysis.png"))
    print(f"\n✓ Player analysis plot saved to: data/processed/eda_player_analysis.png")
    plt.close()

analyze_players(df)


CELL 8: PLAYER-LEVEL ANALYSIS

Player Performance Summary (sorted by number of matches):
--------------------------------------------------------------------------------
                                 Matches  Avg_Passes  Avg_Pass_Comp_Rate  Avg_Carries  Avg_Pressures  Avg_Recoveries  Avg_Possessions  Avg_Possession_Time  Avg_Prog_Passes  Avg_Prog_Carries
player_name                                                                                                                                                                                  
Neymar da Silva Santos Junior         30       52.40                0.80        60.50          11.90            4.13            60.17               362.85             6.90              6.67
Sergio Busquets i Burgos              30       73.03                0.92        58.60          17.03            4.00            65.10               363.52            20.00              2.20
Lionel Andrés Messi Cuccittini        25       68.20                0

In [9]:

def analyze_match_patterns(df):
    """Analyze patterns across matches."""
    print("\n" + "=" * 80)
    print("CELL 9: MATCH-LEVEL PATTERNS")
    print("=" * 80)
    
    # Aggregate by match
    match_stats = df.groupby('match_id').agg({
        'player_id': 'count',
        'passes_attempted': 'sum',
        'carries_attempted': 'sum',
        'pressures_applied': 'sum',
        'ball_recoveries': 'sum',
        'possessions_involved': 'mean',
        'possession_time_seconds': 'mean'
    }).round(2)
    
    match_stats.columns = ['Midfielders', 'Total_Passes', 'Total_Carries', 'Total_Pressures',
                          'Total_Recoveries', 'Avg_Possessions', 'Avg_Possession_Time']
    
    print("\nMatch-Level Statistics:")
    print("-" * 80)
    print(match_stats.describe().round(2).to_string())
    
    # Match activity over time (if we had match dates, we'd use those)
    print(f"\nTotal Matches Analyzed: {len(match_stats)}")
    print(f"Average Midfielders per Match: {match_stats['Midfielders'].mean():.1f}")
    print(f"Total Passes across all matches: {match_stats['Total_Passes'].sum():.0f}")
    print(f"Total Carries across all matches: {match_stats['Total_Carries'].sum():.0f}")
    
    # Save match stats
    match_stats_path = Path("../data/processed/eda_match_stats.csv")
    match_stats.to_csv(match_stats_path)
    print(f"\n✓ Match statistics saved to: {match_stats_path}")

analyze_match_patterns(df)


CELL 9: MATCH-LEVEL PATTERNS

Match-Level Statistics:
--------------------------------------------------------------------------------
       Midfielders  Total_Passes  Total_Carries  Total_Pressures  Total_Recoveries  Avg_Possessions  Avg_Possession_Time
count         39.0         39.00          39.00            39.00             39.00            39.00                39.00
mean           5.0        315.85         298.08            60.33             21.05            58.32               352.63
std            0.0         68.87          72.75            18.32              4.71             5.78               106.42
min            5.0        164.00         151.00            25.00              9.00            40.80               178.06
25%            5.0        262.00         253.00            45.50             18.00            55.40               294.86
50%            5.0        320.00         302.00            62.00             22.00            57.40               354.41
75%            5.

In [10]:
def generate_summary(df, metadata_cols, feature_cols):
    """Generate summary insights."""
    print("\n" + "=" * 80)
    print("CELL 10: SUMMARY AND INSIGHTS")
    print("=" * 80)
    
    print("\nDataset Overview:")
    print("-" * 80)
    print(f"  • Total observations: {len(df):,}")
    print(f"  • Unique players: {df['player_name'].nunique()}")
    print(f"  • Unique matches: {df['match_id'].nunique()}")
    print(f"  • Midfielder types: {df['midfielder_type'].nunique()}")
    print(f"  • Total features: {len(feature_cols)}")
    
    print("\n🎯 Key Insights:")
    print("-" * 80)
    
    # Top insights
    numeric_features = df[feature_cols].select_dtypes(include=[np.number])
    
    # Most variable features
    most_variable = numeric_features.std().nlargest(5)
    print("\n  Most Variable Features (highest std dev):")
    for feat, std_val in most_variable.items():
        print(f"    • {feat}: {std_val:.2f}")
    
    # Features with most zeros
    zero_counts = (numeric_features == 0).sum()
    most_zeros = zero_counts.nlargest(5)
    print("\n  Features with Most Zero Values:")
    for feat, count in most_zeros.items():
        pct = (count / len(df)) * 100
        print(f"    • {feat}: {count} zeros ({pct:.1f}%)")
    
    # Average performance
    print("\n  Average Performance Metrics:")
    print(f"    • Average passes per match: {df['passes_attempted'].mean():.1f}")
    print(f"    • Average pass completion rate: {df['pass_completion_rate'].mean():.2%}")
    print(f"    • Average carries per match: {df['carries_attempted'].mean():.1f}")
    print(f"    • Average pressures per match: {df['pressures_applied'].mean():.1f}")
    
    print("\nEDA Complete!")
    print("=" * 80)
    print("\nAll plots and statistics have been saved to:")
    print("  • data/processed/eda_*.png (visualizations)")
    print("  • data/processed/eda_*.csv (statistics)")

generate_summary(df, metadata_cols, feature_cols)


CELL 10: SUMMARY AND INSIGHTS

Dataset Overview:
--------------------------------------------------------------------------------
  • Total observations: 195
  • Unique players: 12
  • Unique matches: 39
  • Midfielder types: 4
  • Total features: 66

🎯 Key Insights:
--------------------------------------------------------------------------------

  Most Variable Features (highest std dev):
    • carry_distance_total: 158.83
    • possession_time_seconds: 148.25
    • width_variance: 101.53
    • set_piece_involvements: 37.50
    • pressured_touches: 26.61

  Features with Most Zero Values:
    • blocked_passes: 193 zeros (99.0%)
    • blocked_shots: 193 zeros (99.0%)
    • wall_pass_events: 191 zeros (97.9%)
    • fifty_fiftys_won: 191 zeros (97.9%)
    • clearance_followed_by_recovery: 185 zeros (94.9%)

  Average Performance Metrics:
    • Average passes per match: 63.2
    • Average pass completion rate: 85.98%
    • Average carries per match: 59.6
    • Average pressures per matc